<a href="https://colab.research.google.com/github/jasonkjw/daily_coding_commit/blob/main/climate_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import requests
import pandas as pd
import io
from datetime import datetime, timedelta
import time
from google.colab import drive

# 1. 환경 설정
drive.mount('/content/drive')
AUTH_KEY = "FBA6we5nSHuQOsHuZxh7DA"
STATION_ID = "130"
START_YEAR, END_YEAR = 2025, 2025

raw_records = []

print(f"--- {STATION_ID}번 관측소 데이터 수집 시작 ({START_YEAR}-{END_YEAR}) ---")

# 2. 31일 단위 구간 반복 호출 (sfctm3.php 제약 준수)
curr = datetime(START_YEAR, 6, 1)
end_limit = datetime(END_YEAR, 7, 31, 23)

while curr <= end_limit:
    t1 = curr.strftime("%Y%m%d%H%M")
    t2 = (curr + timedelta(days=31)).strftime("%Y%m%d2300")
    if datetime.strptime(t2, "%Y%m%d%H%M") > end_limit:
        t2 = end_limit.strftime("%Y%m%d%H%M")

    url = f"https://apihub.kma.go.kr/api/typ01/url/kma_sfctm3.php?tm1={t1}&tm2={t2}&stn={STATION_ID}&help=0&authKey={AUTH_KEY}"

    try:
        res = requests.get(url, timeout=30)
        # 공백 구분 파싱 (# 주석 제외, 0번: 일시, 11번: TA)
        df = pd.read_csv(io.StringIO(res.text), sep=r'\s+', comment='#', header=None, on_bad_lines='skip')

        for _, row in df.iterrows():
            tm = str(row[0])
            # 관측소, 년, 월, 일, 시간, 기온(TA)만 추출
            raw_records.append([STATION_ID, tm[:4], tm[4:6], tm[6:8], tm[8:10], row[11]])
    except:
        pass

    curr += timedelta(days=31, hours=1)
    time.sleep(0.5)

# 3. 데이터 정리 및 가로 포맷 변환
df_final = pd.DataFrame(raw_records, columns=['STN', 'Year', 'Month', 'Day', 'Hour', 'TA'])
df_final['Hour'] = df_final['Hour'].replace('00', '24') # 00시 표기 보정

# 24시간 가로 배치 (Pivot)
result = df_final.pivot_table(index=['STN', 'Year', 'Month', 'Day'],
                              columns='Hour',
                              values='TA',
                              aggfunc='first').reset_index()

# 4. 저장
save_path = "/content/drive/MyDrive/ULJIN_130_STN_DATA.xlsx"
result.to_excel(save_path, index=False)

print(f"작업 완료. 저장 경로: {save_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- 130번 관측소 데이터 수집 시작 (2025-2025) ---
작업 완료. 저장 경로: /content/drive/MyDrive/ULJIN_130_STN_DATA.xlsx
